## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12_raw")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.when(
        F.col("cid").startswith("NAS"),
        F.substring(F.col("cid"), 4, F.length(F.col("cid")))
    ).otherwise(col("cid"))
)

## Birthdate Validation

In [0]:
df = df.withColumn(
    "BDATE",
    F.when(
        F.col("BDATE") > F.current_date(), None
    ).otherwise(F.col("BDATE"))
)

## Gender Normalization

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.upper(col("GEN")).isin("F", "FEMALE"), "Female")
    .when(F.upper(col("GEN")).isin("M", "MALE"), "Male")
    .otherwise("n/a")
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity Check of Dataframe

In [0]:
display(df.limit(10))

# Writting Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

## Sanity Check of Silver Table

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_customers
LIMIT 10